# UCS420 Cognitive Computing — Assignment 8
## NLP using Python-II
**Topics:** Feature extraction from text, sentiment analysis, text generation

This notebook answers Q1–Q6. Run the cells from top to bottom. The first setup cell installs any missing libraries, and the NLTK setup cell downloads the required language resources.

In [ ]:
# Install required packages if they are not already installed.
%pip install -q nltk scikit-learn textblob wordcloud matplotlib tensorflow

## Q1. Text preprocessing and word frequency

Write a unique paragraph, convert it to lowercase, remove punctuation, tokenize words and sentences, compare Python `split()` with NLTK `word_tokenize()`, remove stopwords, and display word frequencies.

In [ ]:
import re
import nltk
import matplotlib.pyplot as plt
from collections import Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

# Download resources needed by NLTK (safe to run more than once).
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

paragraph = (
    "I enjoy learning about artificial intelligence because it helps computers solve "
    "problems that once required human intelligence. Machine learning can discover "
    "patterns in data and make useful predictions. I also like building small projects "
    "because they turn theoretical ideas into practical tools. Responsible AI is important "
    "because technology should be fair, safe, and helpful to people. In the future, I hope "
    "to create intelligent applications that solve real-world problems."
)

print("Original paragraph:\n", paragraph)

# Convert to lowercase and remove punctuation using re.
clean_text = re.sub(r"[^a-zA-Z0-9\s]", "", paragraph.lower())
print("\nLowercase text without punctuation:\n", clean_text)

# Tokenize into sentences and words.
sentences = sent_tokenize(paragraph)
words_nltk = word_tokenize(clean_text)
print("\nSentence tokens:")
for i, sentence in enumerate(sentences, start=1):
    print(f"{i}. {sentence}")

print("\nWord tokens using NLTK:", words_nltk)

# Compare Python split() and NLTK word_tokenize().
words_split = clean_text.split()
print("\nPython split() tokens:", words_split)
print("Number of tokens using split():", len(words_split))
print("Number of tokens using word_tokenize():", len(words_nltk))
print("\nComparison: split() separates text wherever whitespace occurs. "
      "word_tokenize() is designed for language tokenization and can separate "
      "punctuation into its own tokens. Here, punctuation was removed first, "
      "so their results may be very similar.")

# Remove stopwords.
stop_words = set(stopwords.words("english"))
filtered_words = [word for word in words_nltk if word.isalpha() and word not in stop_words]
print("\nWords after removing stopwords:", filtered_words)

# Frequency distribution excluding stopwords.
freq = Counter(filtered_words)
print("\nWord frequency distribution:")
for word, count in freq.most_common():
    print(f"{word}: {count}")

if freq:
    plt.figure(figsize=(9, 4))
    nltk.FreqDist(filtered_words).plot(15, title="Most Frequent Words (Stopwords Removed)")
    plt.show()

## Q2. Stemming and lemmatization

Use `re.findall()` to extract alphabetic words, remove stopwords, apply Porter stemming and WordNet lemmatization, then compare their outputs.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

# Extract only alphabetic words with re.findall().
alphabetic_words = re.findall(r"\b[a-zA-Z]+\b", paragraph.lower())
print("Alphabetic words:", alphabetic_words)

# Remove stopwords.
words_no_stop = [w for w in alphabetic_words if w not in stop_words]
print("\nAfter removing stopwords:", words_no_stop)

# Stemming.
stemmer = PorterStemmer()
stemmed_words = [stemmer.stem(w) for w in words_no_stop]
print("\nStemmed words:")
print(stemmed_words)

# Lemmatization.
lemmatizer = WordNetLemmatizer()
lemmatized_words = [lemmatizer.lemmatize(w) for w in words_no_stop]
print("\nLemmatized words (default noun POS):")
print(lemmatized_words)

print("\nComparison:")
print("- Stemming cuts words down using rules and may produce non-dictionary forms.")
print("- Lemmatization returns dictionary forms when its vocabulary and part of speech allow it.")
print("- Stemming is often useful when speed and broad matching matter.")
print("- Lemmatization is preferable when readable, linguistically meaningful words matter.")
print("Note: WordNetLemmatizer defaults to nouns; specifying a part of speech can improve results.")

## Q3. Bag of Words and TF-IDF

Three short texts are vectorized with `CountVectorizer` and `TfidfVectorizer`. The top three TF-IDF terms for each text are printed.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

texts = [
    "The new smartphone has an excellent camera and a bright display.",
    "The camera takes sharp photos, and the battery lasts all day.",
    "Electric cars reduce emissions and make city travel quieter."
]

# Bag of Words counts.
count_vectorizer = CountVectorizer(stop_words="english")
bow_matrix = count_vectorizer.fit_transform(texts)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=[f"Text {i}" for i in range(1, len(texts) + 1)]
)
print("Bag of Words representation:")
display(bow_df)

# TF-IDF scores.
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
terms = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=terms,
    index=[f"Text {i}" for i in range(1, len(texts) + 1)]
)
print("TF-IDF scores:")
display(tfidf_df.round(3))

print("\nTop 3 TF-IDF keywords for each text:")
for i, row in enumerate(tfidf_matrix.toarray(), start=1):
    top_indices = row.argsort()[::-1][:3]
    top_terms = [(terms[j], round(row[j], 3)) for j in top_indices if row[j] > 0]
    print(f"Text {i}: {top_terms}")

print("\nInterpretation: Bag of Words records how often terms occur. TF-IDF gives more weight "
      "to terms that are important in a text but less common across the collection.")

## Q4. Jaccard and cosine similarity

Compare two short descriptions of different technologies. Jaccard similarity compares the overlap of unique word sets, while cosine similarity compares their TF-IDF vectors.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

ai_text = (
    "Artificial intelligence learns patterns from data. "
    "AI systems can classify images, understand language, and support decisions. "
    "Machine learning models improve through examples."
)
blockchain_text = (
    "Blockchain stores records in linked blocks. "
    "A distributed network verifies transactions and protects shared records. "
    "Cryptography helps make the ledger difficult to alter."
)

def preprocess_tokens(text):
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())
    tokens = word_tokenize(text)
    return [w for w in tokens if w not in stop_words and w.isalpha()]

tokens_ai = preprocess_tokens(ai_text)
tokens_blockchain = preprocess_tokens(blockchain_text)
set_ai = set(tokens_ai)
set_blockchain = set(tokens_blockchain)

print("AI tokens:", tokens_ai)
print("Blockchain tokens:", tokens_blockchain)

intersection = set_ai & set_blockchain
union = set_ai | set_blockchain
jaccard = len(intersection) / len(union) if union else 0.0
print("\nShared words:", intersection)
print("Jaccard similarity:", round(jaccard, 4))

# TF-IDF cosine similarity.
pair_vectorizer = TfidfVectorizer(stop_words="english")
pair_matrix = pair_vectorizer.fit_transform([ai_text, blockchain_text])
cosine = cosine_similarity(pair_matrix[0:1], pair_matrix[1:2])[0, 0]
print("Cosine similarity:", round(float(cosine), 4))

print("\nAnalysis: These texts discuss different technologies, so both metrics should show "
      "limited similarity. Jaccard is easy to interpret as shared unique words. TF-IDF cosine "
      "also considers term weighting, so it can be more informative when common words and "
      "different term importance need to be accounted for.")

## Q5. Sentiment analysis and word cloud

Use TextBlob to calculate polarity and subjectivity, classify each review as Positive, Negative, or Neutral, and create a word cloud from positive reviews.

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud

reviews = [
    "This laptop is fast, lightweight, and easy to use. The battery life is excellent.",
    "The headphones sound amazing and feel comfortable for long sessions.",
    "The delivery was late and the product stopped working after two days.",
    "The app is frustrating, slow, and full of bugs.",
    "The package arrived today. The box contains the product and a charging cable.",
    "The service was acceptable, but nothing particularly special happened."
]

def classify_sentiment(polarity):
    if polarity > 0.05:
        return "Positive"
    elif polarity < -0.05:
        return "Negative"
    return "Neutral"

review_results = []
for review in reviews:
    blob = TextBlob(review)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    review_results.append({
        "Review": review,
        "Polarity": round(polarity, 3),
        "Subjectivity": round(subjectivity, 3),
        "Sentiment": classify_sentiment(polarity)
    })

reviews_df = pd.DataFrame(review_results)
display(reviews_df)

positive_reviews = [r for r in reviews if classify_sentiment(TextBlob(r).sentiment.polarity) == "Positive"]
positive_text = " ".join(positive_reviews)

if positive_text.strip():
    cloud = WordCloud(width=900, height=450, background_color="white",
                      collocations=False).generate(positive_text)
    plt.figure(figsize=(12, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Word Cloud of Positive Reviews")
    plt.show()
else:
    print("No positive reviews were detected, so a word cloud cannot be generated.")

print("Polarity ranges from -1 (negative) to +1 (positive).")
print("Subjectivity ranges from 0 (more objective) to 1 (more subjective).")
print("The thresholds above are a simple rule for this assignment; real applications may need tuning.")

## Q6. Text generation using a simple LSTM

Use an approximately 100-word paragraph as training data, tokenize it with Keras `Tokenizer`, create input sequences, train a small LSTM model, and generate 2–3 lines from a seed phrase.

**Note:** Text generation quality depends heavily on the size of the training corpus. With only one short paragraph, the output is a demonstration and may be repetitive. TensorFlow may take a little time to install or train.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

tf.keras.utils.set_random_seed(42)

training_text = (
    "Technology helps people solve everyday problems. Artificial intelligence learns from data "
    "and finds useful patterns. Machine learning supports prediction and decision making. "
    "Good software should be reliable, safe, and easy to use. Developers test their programs "
    "and improve them over time. Data helps systems understand trends and provide useful results. "
    "Creative ideas become valuable when people build and test practical projects. Responsible "
    "technology should respect privacy and help communities. Learning new skills takes patience "
    "and practice. Small experiments help learners understand difficult concepts. A clear goal "
    "makes a project easier to plan. Strong applications combine useful features with simple design."
)

print("Approximate number of words:", len(training_text.split()))

tokenizer = Tokenizer()
tokenizer.fit_on_texts([training_text])
total_words = len(tokenizer.word_index) + 1
print("Vocabulary size:", total_words)

# Create progressively longer input sequences.
input_sequences = []
for line in training_text.split("."):
    line = line.strip()
    if not line:
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i + 1])

max_sequence_len = max(len(seq) for seq in input_sequences)
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding="pre")

X = padded_sequences[:, :-1]
y = padded_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("Number of training sequences:", len(X))
print("Input shape:", X.shape)
print("Target shape:", y.shape)

model = Sequential([
    Embedding(input_dim=total_words, output_dim=32, input_length=max_sequence_len - 1),
    LSTM(64),
    Dense(total_words, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.summary()

history = model.fit(X, y, epochs=150, verbose=0)
print("Training completed.")
print("Final training accuracy:", round(float(history.history["accuracy"][-1]), 3))

In [ ]:
def generate_text(seed_text, next_words=20):
    generated = seed_text.lower()
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([generated])[0]
        sequence = pad_sequences([sequence], maxlen=max_sequence_len - 1, padding="pre")
        probabilities = model.predict(sequence, verbose=0)[0]
        next_index = int(np.argmax(probabilities))
        if next_index == 0:
            break
        next_word = tokenizer.index_word.get(next_index, "")
        if not next_word:
            break
        generated += " " + next_word
    return generated

seed = "technology helps"
print("Generated line 1:", generate_text(seed, 12))
print("Generated line 2:", generate_text("machine learning", 12))
print("Generated line 3:", generate_text("good software", 12))

## Conclusion

- **Q1:** Cleaned and tokenized text, removed stopwords, and inspected word frequencies.
- **Q2:** Compared Porter stemming with WordNet lemmatization.
- **Q3:** Created Bag of Words and TF-IDF representations and extracted top keywords.
- **Q4:** Calculated Jaccard and cosine similarity for two technology descriptions.
- **Q5:** Measured review polarity and subjectivity, classified sentiment, and generated a word cloud.
- **Q6:** Built and trained a small LSTM next-word model and generated text from seed phrases.